# E2 - Ingesta y procesamiento

En este notebook procesamos las mediciones y las comidas de la aplicación. Unificamos los formatos, depuramos registros repetidos, construimos las tablas diarias y organizamos los resultados para analizarlos. También ejecutamos las tres consultas analíticas definidas en E1 y comparamos los resultados obtenidos con CSV y Parquet.

## 1. Configuración

Configuramos el notebook para que localice automáticamente la carpeta del proyecto y los datos disponibles. De ese modo, el análisis depende de la estructura del repositorio y no de una ubicación personal del equipo.

## Cobertura nocturna y cambios respecto de E1

Conservamos como nula la frecuencia cardíaca en reposo de cada usuario y día cuando las mediciones nocturnas no alcanzan el mínimo definido para el reloj. En el universo de 216 usuarios identificamos 41 días sin lecturas nocturnas válidas, de un total de 11 864 días elegibles.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'pipeline_e2.py').exists()), None)
assert ROOT is not None, 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(ROOT / 'src'))
from pipeline_e2 import run_pipeline
ROOT

WindowsPath('.')

## 2. Procesamiento de los datos

Procesamos 1 449 archivos comprimidos de mediciones. Conservamos intactos los archivos originales y guardamos los resultados procesados en datos/processed/e2/, una carpeta excluida del repositorio.

In [2]:
reporte_e2 = run_pipeline(ROOT)
reporte_e2['counts']

Procesados 120/1449 archivos de mediciones; filas=1,100,720
Procesados 240/1449 archivos de mediciones; filas=2,274,361
Procesados 360/1449 archivos de mediciones; filas=3,472,520
Procesados 480/1449 archivos de mediciones; filas=4,694,333
Procesados 600/1449 archivos de mediciones; filas=5,968,386
Procesados 720/1449 archivos de mediciones; filas=7,316,235
Procesados 840/1449 archivos de mediciones; filas=8,683,430
Procesados 960/1449 archivos de mediciones; filas=10,047,112
Procesados 1080/1449 archivos de mediciones; filas=11,418,304
Procesados 1200/1449 archivos de mediciones; filas=12,766,404
Procesados 1320/1449 archivos de mediciones; filas=14,130,000
Procesados 1440/1449 archivos de mediciones; filas=15,497,476
{
  "pipeline": "E2 local, Python estándar + DuckDB",
  "data_root": "E:\\Maestria Pontifica Catolica de Chile\\Cursos en desarrollo\\bigdata\\Proyecto\\datos_proyecto_INF3862",
  "output_dir": "E:\\Maestria Pontifica Catolica de Chile\\Cursos en desarrollo\\bigdata\\Pro

{'measurement_rows_raw': 15540132, 'measurement_rows_unique': 15194533, 'heart_rate_null': 349931, 'duplicate_device_timestamp': 345599, 'received_over_1h_late': 8957451, 'nutrient_join_available': 1, 'recipe_rows_parquet_read': 522517, 'meals_without_recipe_match': 48, 'users_in_device_registry': 300, 'users_with_resting_hr': 216, 'days_in_utc_window': 61, 'measurement_files': 1449, 'meal_rows': 22106, 'meal_recipe_ids_distinct': 20702, 'recipe_parquet_present': 1, 'invalid_sessions': 0, 'sessions_outside_binding': 0, 'session_rows_raw': 4840, 'fact_usuario_dia_rows': 18600, 'fact_usuario_reloj_dia_rows': 16330, 'rows_with_resting_hr': 11823, 'rows_with_nutrient_join': 12590, 'elapsed_seconds': 229.92}

## 3. Tablas y consultas analíticas de E1

Generamos las tablas diarias por usuario y por reloj, junto con las dimensiones de usuario, dispositivo y fecha. Guardamos la tabla diaria en formato Parquet comprimido y organizada por fecha local. También obtuvimos los archivos CSV de las consultas 4, 5 y 6 definidas en E1. El reporte de calidad reúne las reglas aplicadas, los conteos, la comparación de tiempos y una proyección de escala.

In [3]:
import csv, json
OUT = ROOT / 'datos' / 'processed' / 'e2'
with (OUT / 'fact_usuario_dia.csv').open(encoding='utf-8-sig', newline='') as f:
    muestra = list(csv.DictReader(f))[:5]
display(muestra)
print(json.dumps(reporte_e2, ensure_ascii=False, indent=2))
for nombre, detalle in reporte_e2['analytical_queries'].items():
    print(nombre, 'filas:', detalle['filas_resultado'])
    display(detalle['muestra'])

[{'usuario_id': '102058', 'fecha_local': '2007-12-15', 'pasos': '0.0', 'calorias_gastadas': '0.0', 'calorias_sesion': '0.0', 'minutos_sesion': '0.0', 'numero_comidas': '0', 'porciones': '0.0', 'fc_reposo': '', 'n_mediciones_fc_reposo': '0', 'calorias_consumidas': '', 'sodio': '', 'grasa_saturada': '', 'azucar': '', 'fibra': '', 'proteina': ''}, {'usuario_id': '102058', 'fecha_local': '2007-12-16', 'pasos': '5395.0', 'calorias_gastadas': '1920.8', 'calorias_sesion': '0.0', 'minutos_sesion': '0.0', 'numero_comidas': '1', 'porciones': '3.0', 'fc_reposo': '63.613', 'n_mediciones_fc_reposo': '31', 'calorias_consumidas': '1804.5', 'sodio': '1134.9', 'grasa_saturada': '22.200000000000003', 'azucar': '3.5999999999999996', 'fibra': '18.0', 'proteina': '196.20000000000002'}, {'usuario_id': '102058', 'fecha_local': '2007-12-17', 'pasos': '4203.0', 'calorias_gastadas': '1829.72', 'calorias_sesion': '0.0', 'minutos_sesion': '0.0', 'numero_comidas': '1', 'porciones': '3.0', 'fc_reposo': '65.875', 'n

## 4. Decisiones y límites

- Eliminamos duplicados usando el identificador del reloj y la marca de tiempo de cada medición. Convertimos las fechas de UTC a la zona horaria local del dispositivo.
- Calculamos la frecuencia cardíaca en reposo con lecturas de 02:00 a 04:59, fuera de las sesiones de actividad, y exigimos un mínimo por hora según el modelo del reloj: 30 lecturas para Pulso S y 6 para FitBand 2.
- Tratamos las comidas como eventos registrados en la aplicación; no las consideramos un registro completo de la dieta.
- Vinculamos la información nutricional mediante el identificador de receta y las porciones. No completamos los nutrientes ausentes ni las comidas sin una receta coincidente.
- Consideramos que los datos de los relojes son sintéticos. Por ello, describimos asociaciones y no las interpretamos como efectos causales ni como conclusiones clínicas.

## 5. Comparación de CSV y Parquet

Comprobamos que una misma consulta devuelve los mismos resultados en CSV y Parquet. En nuestra ejecución local, la consulta tardó 0,128 s con CSV y 0,170 s con Parquet; estos tiempos incluyen el inicio del proceso DuckDB y describen una sola ejecución en este equipo. El conjunto Parquet ocupó menos espacio que el CSV, aunque el tamaño de los archivos no equivale a los bytes que la consulta leyó. Estimamos el volumen a tres años y cien veces el actual mediante una proyección lineal; no medimos ese escenario en un sistema distribuido ni garantizamos su rendimiento.

## E3: tabla para el modelamiento y modelo de referencia

A partir de las tablas generadas en E2 construimos la tabla para el modelamiento. Comparamos un modelo de referencia basado en la mediana con una regresión lineal, usando una separación temporal para evaluar fechas posteriores.

In [4]:
import json
from modelamiento_e3 import run_model
metricas_e3 = run_model(ROOT)
print(json.dumps(metricas_e3, ensure_ascii=False, indent=2))

{
  "objetivo": "frecuencia cardíaca en reposo del día siguiente (bpm)",
  "ventana_features": "7 días anteriores; no incluye frecuencia cardíaca como predictor",
  "particion": "temporal global: train hasta 2008-02-02; test posterior. Los usuarios pueden aparecer en ambos periodos; ninguna variable posterior a fecha_features_hasta se usa como predictor",
  "fecha_corte_temporal": "2008-02-02",
  "features": [
    "pasos_7d_promedio",
    "minutos_sesion_7d_promedio",
    "comidas_7d",
    "porciones_7d",
    "calorias_7d",
    "sodio_7d",
    "grasa_saturada_7d",
    "azucar_7d",
    "fibra_7d",
    "proteina_7d"
  ],
  "nutrition_feature_policy": "7-day sums over matched, logged meals; no-meal days contribute zero recorded intake; windows with meal events missing nutrient joins are excluded",
  "n_rows_total": 10710,
  "n_rows_train": 8445,
  "n_rows_test": 2265,
  "n_users_train": 216,
  "n_users_test": 216,
  "usuarios_compartidos_train_test": 216,
  "baseline_mediana_train": 60.96